# 10 · `Command`: update and route from inside a node

**Goal:** by the end you can make a node both update the state **and** choose the next node, by returning a
`Command`, and you know when to use that instead of a conditional edge.

## The idea in plain English

In lesson 05 two separate pieces did two jobs. The `parse` node **updated** the state (`a`, `op`, `b`), and then a
router function **decided** where to go next. Often the same code knows both things at once: `parse` has just read the
operator, so it already knows the next node.

A **`Command`** lets one node do both. Instead of a dict, the node returns
`Command(update={...}, goto="multiply")`. The `update` is merged into the state exactly like a normal return value,
and `goto` names the next node. To let LangGraph draw the possible routes, we annotate the return type as
`Command[Literal["add", "subtract", ...]]`.

**Analogy:** in lesson 05 a clerk filled in the form and a separate sorter read it and chose the next desk. Now the
clerk fills in the form *and* writes "next: multiply desk" on it, so no sorter is needed.

## Picture

Same routes as lesson 05, but the dashed arrows now come straight from `parse`'s `Command`, with no router function.

```mermaid
flowchart LR
    S([START]) --> parse
    parse -. "Command(goto=...)" .-> add & subtract & multiply & divide & error
    add & subtract & multiply & divide --> format
    format --> E([END])
    error --> E
```

## Step 1: State and the operation nodes

Exactly as in lesson 05. Each operation node prints its name so you can see the route taken.

In [1]:
from typing import Literal, TypedDict

from langgraph.graph import StateGraph, START, END


class CalcState(TypedDict):
    expression: str
    a: float
    op: str
    b: float
    result: float
    answer: str

The operation nodes, exactly as in lesson 05. Each prints its name so the route is visible.

In [2]:
def add(state: CalcState) -> dict:
    print("  -> add")
    return {"result": state["a"] + state["b"]}


def subtract(state: CalcState) -> dict:
    print("  -> subtract")
    return {"result": state["a"] - state["b"]}


def multiply(state: CalcState) -> dict:
    print("  -> multiply")
    return {"result": state["a"] * state["b"]}


def divide(state: CalcState) -> dict:
    print("  -> divide")
    return {"result": state["a"] / state["b"]}

`format_answer` builds the sentence for a successful calculation.

In [3]:
def format_answer(state: CalcState) -> dict:
    return {"answer": f"{state['expression']} = {state['result']:g}"}

## Step 2: What a `Command` is

`Command` comes from `langgraph.types`. Like `Send`, making one doesn't do anything yet. It's a description that
LangGraph carries out when a node returns it: "merge this `update`, then go to `goto`".

In [4]:
from langgraph.types import Command

cmd = Command(update={"a": 3.0, "op": "*", "b": 4.0}, goto="multiply")
print(cmd)
print("update:", cmd.update)
print("goto:  ", cmd.goto)

Command(update={'a': 3.0, 'op': '*', 'b': 4.0}, goto='multiply')
update: {'a': 3.0, 'op': '*', 'b': 4.0}
goto:   multiply


## Step 3: `parse` returns a `Command`

`parse` now does what `parse` + `route` did in lesson 05. The return type
`Command[Literal["add", "subtract", "multiply", "divide", "error"]]` lists every node `goto` might name. It does the
same job the `Literal` hint did on the router: LangGraph reads it to check the routes and draw the diagram.

In [5]:
OPERATIONS = {"+": "add", "-": "subtract", "*": "multiply", "/": "divide"}


def parse(state: CalcState) -> Command[Literal["add", "subtract", "multiply", "divide", "error"]]:
    left, op, right = state["expression"].split()
    return Command(
        update={"a": float(left), "op": op, "b": float(right)},
        goto=OPERATIONS.get(op, "error"),
    )


print(parse({"expression": "6 * 7"}))

Command(update={'a': 6.0, 'op': '*', 'b': 7.0}, goto='multiply')


## Step 4: A `Command` can also go to `END`

The `error` node uses a `Command` too. It sets the `answer` and goes straight to `END`. That means it needs no
`add_edge("error", END)`.

In [6]:
def error(state: CalcState) -> Command[Literal[END]]:
    print("  -> error")
    return Command(
        update={"answer": f"Sorry, I don't know the operator {state['op']!r}. Use +, -, * or /."},
        goto=END,
    )

## Step 5: Build the graph

Compare with lesson 05. There's **no** `add_conditional_edges` and no edge out of `parse` or `error`. Their
`Command`s do the routing. The fixed edges are still there for the parts that never change: `START → parse`, each
operation `→ format`, and `format → END`.

In [7]:
builder = StateGraph(CalcState)
builder.add_node("parse", parse)
for name, fn in [("add", add), ("subtract", subtract), ("multiply", multiply), ("divide", divide)]:
    builder.add_node(name, fn)
    builder.add_edge(name, "format")
builder.add_node("format", format_answer)
builder.add_node("error", error)

builder.add_edge(START, "parse")
builder.add_edge("format", END)

graph = builder.compile()
print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	parse(parse)
	add(add)
	subtract(subtract)
	multiply(multiply)
	divide(divide)
	format(format)
	error(error)
	__end__([<p>__end__</p>]):::last
	__start__ --> parse;
	add --> format;
	divide --> format;
	error -.-> __end__;
	multiply --> format;
	parse -.-> add;
	parse -.-> divide;
	parse -.-> error;
	parse -.-> multiply;
	parse -.-> subtract;
	subtract --> format;
	format --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## Step 6: Run it

The same results as lesson 05: one route per run, and `%` goes to `error`.

In [8]:
for expression in ["3 + 4", "10 - 2.5", "6 * 7", "10 / 4", "2 % 3"]:
    print(expression)
    print("  answer:", graph.invoke({"expression": expression})["answer"])

3 + 4
  -> add
  answer: 3 + 4 = 7
10 - 2.5
  -> subtract
  answer: 10 - 2.5 = 7.5
6 * 7
  -> multiply
  answer: 6 * 7 = 42
10 / 4
  -> divide
  answer: 10 / 4 = 2.5
2 % 3
  -> error
  answer: Sorry, I don't know the operator '%'. Use +, -, * or /.


## Step 7: Why the `Command[Literal[...]]` hint matters

Like a router in lesson 05, a node returning `Command` without the hint still **runs** correctly, but LangGraph can't
see where it might go. The diagram loses every dashed arrow from `parse` and shows a wrong `parse --> __end__`.

To try other `parse` nodes without copying the wiring, we put Step 5's wiring in a function. `destinations` is
explained just below. Its default, `None`, means "not given".

In [9]:
def build_command_graph(parse_node, destinations=None):
    builder = StateGraph(CalcState)
    builder.add_node("parse", parse_node, destinations=destinations)
    for name, fn in [("add", add), ("subtract", subtract), ("multiply", multiply), ("divide", divide)]:
        builder.add_node(name, fn)
        builder.add_edge(name, "format")
    builder.add_node("format", format_answer)
    builder.add_node("error", error)
    builder.add_edge(START, "parse")
    builder.add_edge("format", END)
    return builder.compile()


def parse_arrows(graph):
    """The diagram lines that start at parse."""
    return [line.strip() for line in graph.get_graph().draw_mermaid().splitlines() if line.strip().startswith("parse ")]

Now the same `parse` code **without** the return type hint. The answer is still right, but look at the arrows that leave `parse`.

In [10]:
def parse_no_hint(state: CalcState):  # same code, no return type
    left, op, right = state["expression"].split()
    return Command(update={"a": float(left), "op": op, "b": float(right)}, goto=OPERATIONS.get(op, "error"))


unhinted = build_command_graph(parse_no_hint)
print(unhinted.invoke({"expression": "3 + 4"})["answer"])
print(parse_arrows(unhinted))

  -> add
3 + 4 = 7
['parse --> __end__;']


If you can't (or don't want to) write the hint, `add_node` accepts `destinations=(...)`, a tuple of the node names
`goto` might use. It fixes the diagram in the same way:

In [11]:
fixed = build_command_graph(parse_no_hint, destinations=("add", "subtract", "multiply", "divide", "error"))
print(parse_arrows(fixed))

['parse -.-> add;', 'parse -.-> divide;', 'parse -.-> error;', 'parse -.-> multiply;', 'parse -.-> subtract;']


## Conditional edge or `Command`?

| | Conditional edge (lesson 05) | `Command` (this lesson) |
|---|---|---|
| Who decides the next node | A separate router function | The node itself |
| Can the decision also update the state? | No, routers only read the state | Yes, `update` and `goto` together |
| Where you see the routing | In the wiring (`add_conditional_edges`) | In the node's return type |
| Good for | Routing that's a separate, reusable rule | A node that works something out and knows where to go next |

Both are fine. A simple rule: if you'd otherwise write a node that stores a value only so a router can read it a
moment later, use `Command`. If the decision is a rule you want to keep separate, or reuse after several nodes, use
a conditional edge. `Command` is also used to resume after a pause (lesson 14).

## What just happened

- **Step 2** printed a `Command` with `update={'a': 3.0, 'op': '*', 'b': 4.0}` and `goto='multiply'`.
- **Step 3** called `parse` directly. For `"6 * 7"` it returned a `Command` with the parsed values and
  `goto='multiply'`.
- **Step 5** built the graph with no router and no edge out of `parse`. The diagram still shows dashed arrows to all
  five nodes, taken from the `Command[Literal[...]]` hint.
- **Step 6** gave the same answers as lesson 05, with `2 % 3` going to `error` and from there to `END`.
- **Step 7** removed the hint. The answer was still right, but the diagram only showed `parse --> __end__`. Adding
  `destinations=(...)` brought the dashed arrows back.

## Common mistakes

1. **A `Command` plus a fixed edge from the same node.** If `parse` returns `Command(goto="add")` and you also wrote
   `add_edge("parse", "format")`, there's **no error**. Both `add` and `format` run. `goto` adds a route; it doesn't
   replace the fixed edges.
   Fix: when a node routes with `Command`, remove the fixed edges out of it.
2. **No return type hint.** The graph runs, but the diagram is wrong and LangGraph can't check the routes (Step 7).
   Fix: annotate `-> Command[Literal[...]]`, or pass `destinations=(...)` to `add_node`.
3. **A typo in `goto`.** `goto="ad"` gives **no error**, even with a hint. LangGraph only logs
   `wrote to unknown channel branch:to:ad, ignoring it`, and the run quietly stops after `parse`.
   Fix: check the name. Building `goto` from a dict, like `OPERATIONS.get(op, "error")`, keeps names in one place.

## Try it

1. Make `parse` send `/` with `b == 0` to a new `divide_by_zero` node (as in lesson 05's exercise), using only the
   `Command`. Add the name to the hint, and have `divide_by_zero` return a `Command` to `END`.
2. Change `format_answer` to return `Command(update={"answer": ...}, goto=END)` and remove
   `builder.add_edge("format", END)`. Does the graph still work? What does the diagram show for `format`?

Answers are in [`solutions/exercises_solutions.ipynb`](../solutions/exercises_solutions.ipynb).

## Key terms

New terms from this lesson, also in [GLOSSARY.md](../GLOSSARY.md):

- **`Command`**: a value a node can return to update the state (`update=`) and choose the next node (`goto=`) at once.
- **`goto`**: the part of a `Command` naming the next node (or `END`).
- **`Command[Literal[...]]`**: the return type hint that lists every node a `Command` might go to.
- **`destinations`**: an `add_node` option listing a node's possible `goto` targets, an alternative to the hint.